# 08 — Modifying Tilings: Surgery and Coordinate Transforms

Conway operators are global; sometimes you want a scalpel. This notebook collects the ad-hoc modification primitives that [`eucare.half`](../reference/eucare/half.md) exposes:

- finding faces / edges by predicate,
- deleting faces, edges, and (degree-2) vertices,
- subdividing edges and faces,
- transforming all positions in one go (stretch, warp).

We highlight every change by writing a `color_key` and showing the graph before and after.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    reciprocal_figures,
    rendering,
)
from eucare.rendering import multi_show


## Finding by predicate

All collections are plain Python sets. Filter with a list comprehension and mark hits with a colour key.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.t_4_6_12(), rings=2)
G.recompute_lengths_and_angles()
hexagons = [f for f in G.faces if f.order() == 6]
for f in hexagons:
    f['color_key'] = (1.0, 0.6, 0.0, 0.8)
print(f'{len(hexagons)} hexagons found')
G.show(render_faces=True, face_inset=0.05, render_vertices=False)


## Deleting faces

`G.delete_faces(set_of_faces)` removes faces and the half-edges that bordered only them. The neighbouring faces become adjacent to a new piece of border.

In [ ]:
G_before = G.copy()
to_delete = [f for f in G.faces if f.order() == 6
             and np.linalg.norm(f.midpoint()) < 2.5]
G.delete_faces(set(to_delete))
multi_show([G_before, G],
           titles=['before delete_faces', f'after (-{len(to_delete)} hex)'],
           render_faces=True, face_inset=0.05, render_vertices=False)


## Deleting an edge merges two faces

`G.delete_edge(h)` removes the halfedge and its reverse, merging the two adjacent faces into one. We mark the merged face with a single colour to make the result visible.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=3)
G.recompute_lengths_and_angles()
central = G.central_face()
h = next(h for h in central.halfedge_iter()
         if not h.on_border() and not h.rev.on_border())
G_before = G.copy()
h['color_key'] = h.rev['color_key'] = (0.85, 0.1, 0.1, 1.0)
h['line_width'] = h.rev['line_width'] = 5
G.delete_edge(h)
merged = next(f for f in G.faces if f.order() > 6)
merged['color_key'] = (0.6, 0.85, 0.4, 0.9)
multi_show([G_before, G],
           titles=['edge to delete (red)', 'merged face (green)'],
           render_faces=True, face_inset=0.05, render_vertices=False)


## Subdividing edges and faces

- `G.subdivide_edge(h)` inserts a new vertex on `h` (at the midpoint by default) and returns `(h_new, v_new)`.
- `G.subdivide_face(f, v1, v2)` adds a new edge between two of `f`'s vertices, splitting it. Returns `(h_new, f_new)`.

Below we subdivide a hexagon's diagonal.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G.recompute_lengths_and_angles()
f = G.central_face()
vs = list(f.vertex_iter())
G_before = G.copy()
h_new, f_new = G.subdivide_face(f, vs[0], vs[3])
f_new['color_key'] = (0.4, 0.7, 1.0, 0.9)
f['color_key'] = (1.0, 0.5, 0.1, 0.9)
multi_show([G_before, G],
           titles=['before subdivide_face', 'after — split into halves'],
           render_faces=True, face_inset=0.05, render_vertices=False)


## Coordinate transformations

Vertex positions live at `v['pos']`. Mutating them in place and then calling `G.recompute_lengths_and_angles()` is the standard way to bend / stretch / warp a graph.

Below we apply a horizontal shear and a sinusoidal vertical warp.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=3)
G.recompute_lengths_and_angles()
G_orig = G.copy()

for v in G.vertices:
    x, y = v['pos']
    v['pos'] = np.array([x + 0.3 * y, y + 0.4 * np.sin(x)])
G.recompute_lengths_and_angles()

multi_show([G_orig, G],
           titles=['original', 'shear + sin warp'],
           face_inset=0.05, render_vertices=False)


## Surgery before SRG

All of these primitives produce ordinary half-edge graphs, so downstream pipelines such as SRG just work. Here we delete a ring of hexagons and then run `srg_pipeline` on the result — the resulting CP wraps cleanly around the hole.

In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, shrink_rotate_pattern


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = shrink_rotate_pattern(G)
    SRG.recompute_lengths_and_angles()
    return SRG


In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=3)
G.recompute_lengths_and_angles()
# delete a few non-adjacent hexagons (keep face connectivity for BFS)
to_delete = [f for f in G.faces
             if 1.5 < np.linalg.norm(f.midpoint()) < 2.0][::2]
G.delete_faces(set(to_delete))
G.recompute_lengths_and_angles()
SRG = srg_pipeline(G)
multi_show([G, SRG],
           titles=[f'tiling minus {len(to_delete)} faces', 'SRG of the holey tiling'],
           face_inset=0.04, render_vertices=False)


## What we didn't cover

- `G.glue_v2v(...)`, `G.glue_e2e(...)`, `G.add_graph(...)` — for combining graphs and stitching seams. See the legacy [`Alternating Flagstones`](Alternating%20Flagstones.ipynb) notebook for an in-the-wild example of these used to extend a CP boundary.
- `G.cut_along_halfplane(...)` and friends in [`eucare.cutting`](../reference/eucare/cutting.md) — for unfolding a CP onto a flat sheet.
- `G.convert_to_euclidean()` — flatten a curved-geometry graph into a Euclidean one (uses `geometry.to_euclidean` per vertex).